# MonopolyZero — ASU and Fixed A/B/C evaluation

Runs seat-balanced `ppo-plus-v2` matches for the exact Fixed A/B/C table, three-copy ASU, and the mixed ASU/DealMaker/Gambler lineup. Upload the candidate as `/content/monopolyzero-candidate.pt` or provide its path in the job JSON.

In [ ]:
from pathlib import Path
import hashlib, json, os, re, sys, tarfile, urllib.request

CONTENT = Path(os.environ.get('MONOPOLYZERO_CONTENT', '/content'))
JOB = {
    'commit': '50cb613a35b03d920ca4b1048dce29f25e43c476',
    'candidate': str(CONTENT / 'monopolyzero-candidate.pt'),
    'games': 200,
    'seed_base': 3100000,
}
job_path = CONTENT / 'monopolyzero-evaluate-job.json'
if job_path.exists():
    JOB.update(json.loads(job_path.read_text()))
assert re.fullmatch(r'[0-9a-f]{40}', JOB['commit'])
assert int(JOB['games']) > 0 and int(JOB['games']) % 4 == 0
CANDIDATE = Path(JOB['candidate'])
REPORT_PATH = CONTENT / 'monopolyzero-evaluation.json'
print(json.dumps(JOB, indent=2, sort_keys=True))

In [ ]:
repository_archive = CONTENT / 'DeepRL_Monopoly.tar.gz'
if repository_archive.exists():
    with tarfile.open(repository_archive, 'r:gz') as archive:
        archive.extractall(CONTENT, filter='data')
    REPOSITORY_ROOT = CONTENT / 'DeepRL_Monopoly'
else:
    repository_archive = CONTENT / f"DeepRL_Monopoly-{JOB['commit']}.tar.gz"
    urllib.request.urlretrieve(
        f"https://codeload.github.com/Darkosxl/DeepRL_Monopoly/tar.gz/{JOB['commit']}",
        repository_archive,
    )
    with tarfile.open(repository_archive, 'r:gz') as archive:
        roots = {Path(member.name).parts[0] for member in archive.getmembers() if member.name}
        assert len(roots) == 1
        archive.extractall(CONTENT, filter='data')
    REPOSITORY_ROOT = CONTENT / roots.pop()
assert (REPOSITORY_ROOT / 'monopoly_bench').is_dir()
assert CANDIDATE.is_file(), f'Missing candidate: {CANDIDATE}'
sys.path.insert(0, str(REPOSITORY_ROOT))
print(f'Repository: {REPOSITORY_ROOT} | Candidate: {CANDIDATE}')

In [ ]:
from monopoly_bench.adapters import ASUAdapter, FixedAdapter
from monopoly_bench.config import BenchmarkConfig
from monopoly_bench.ladder import evaluate_baseline
from monopoly_bench.model import MonopolyZeroNet
from monopoly_game_engine.agents_fixed import FPAgentA, FPAgentB, FPAgentC

model = MonopolyZeroNet.load_inference(CANDIDATE)
assert model.policy_head.out_features == 2958
config = BenchmarkConfig()
matchups = {
    'fixed_abc': (FixedAdapter(FPAgentA), FixedAdapter(FPAgentB), FixedAdapter(FPAgentC)),
    'asu_value_x3': ASUAdapter(),
    'asu_dealmaker_gambler': (ASUAdapter(), FixedAdapter(FPAgentB), FixedAdapter(FPAgentC)),
}
print(list(matchups))

In [ ]:
summaries = {}
for index, (name, opponents) in enumerate(matchups.items()):
    summary = evaluate_baseline(
        CANDIDATE, opponents, games=int(JOB['games']), config=config,
        seed_base=int(JOB['seed_base']) + index * 10000,
    )
    summaries[name] = summary.as_dict()
    print(name, json.dumps(summaries[name], indent=2, sort_keys=True))
report = {
    'schema': 1, 'ruleset': config.ruleset, 'candidate': str(CANDIDATE),
    'candidate_sha256': hashlib.sha256(CANDIDATE.read_bytes()).hexdigest(),
    'games_per_matchup': int(JOB['games']), 'seed_base': int(JOB['seed_base']),
    'matchups': summaries,
}
temporary = REPORT_PATH.with_suffix('.tmp')
temporary.write_text(json.dumps(report, indent=2, sort_keys=True) + '\n')
temporary.replace(REPORT_PATH)
print(f'Report: {REPORT_PATH}')